# SANPO-Synthetic validation split (official train split, chest + head)

**Purpose:** a validation set of synthetic walks to watch the synthetic pretraining stages (experiments 6 and 7) and to
choose their checkpoint. The final models are still chosen on the real chest validation set. The official synthetic
test split stays untouched.

**Method:** the same as the real chest split (`../../chest/validation_split/validation_split.ipynb`): whole walks,
rules that every candidate must keep, random candidate splits scored by distances between validation and train, then
swaps from the best candidates. **Five differences** (synthetic data is different):

1. **No walk descriptions** (no `description.json`): no weather, environment or crowd rules; the rules use classes only.
2. **No human / machine labels:** every frame is rendered, so that rule is gone.
3. **Two cameras:** each camera's validation holds about 10% of that camera's frames, so validation has both cameras in
   the same proportion as train, and chest and head can be scored separately.
4. **Frame rates:** walks have 50, 143 or 167 frames (5, 14.28, 33.33 FPS); each camera's validation keeps about the
   same number of walks of each frame rate as 10% of its walks.
5. **Shared locations:** many synthetic walks pass the same spots of a few virtual towns (found in section 3). Walks
   whose paths come within `LOCATION_RADIUS_M` of each other form a **location group**, and a group goes to validation
   or train **as a whole**, so no validation walk retraces a train walk's spot.

**Input:** the cached results of `../explore_sanpo_synthetic.ipynb` (`../results/index_all.csv`,
`../results/class_pixels_per_frame.csv`); nothing is read from Modal. **Output:** `results/validation_walks.txt` (read by
`../prepare_sanpo_synthetic_modal.py`, which then writes `val.txt` and `data.yaml`) and `results/validation_split.json`.

## 1. Settings

In [ ]:
from pathlib import Path

INDEX_FILE = Path("../results/index_all.csv").resolve()                 # one row per frame (explore notebook, section 4)
CLASS_PIXELS_FILE = Path("../results/class_pixels_per_frame.csv").resolve()  # pixels per class per frame (section 6)
RESULTS_DIR = Path("results").resolve()                                  # the chosen walks are saved here

# Rules (per camera; every candidate must keep all of them)
VALIDATION_SHARE = 0.10              # target share of each camera's frames in validation
FRAME_SHARE_RANGE = (0.08, 0.12)     # each camera's validation must hold between 8% and 12% of its frames
FRAMES_TO_FPS = {50: 5.0, 143: 14.28, 167: 33.33}   # frames per walk -> frame rate (explore notebook, section 5)
FPS_WALK_TOLERANCE = 0.5             # validation walks per frame rate: within +-50% (and at least +-1) of 10% of them
REQUIRED_CLASS_MIN_WALK_SHARE = 0.05 # a class in at least 5% of a camera's walks must appear in its validation ...
MIN_FRAMES_IN_VALIDATION = 10        # ... in at least this many validation frames
MIN_RELATIVE_FRAME_SHARE = 0.5       # ... and its share of validation frames >= this x its share in train
MAX_RELATIVE_FRAME_SHARE = 2.0       # ... and <= this x its share in train
MAX_SHARE_OF_CLASS_WALKS = 1 / 3     # validation may take at most this share of the walks that contain a class
IGNORED_CLASSES = ["unlabeled", "bus stop"]   # not trained, so not used in the rules or the scores
LOCATION_RADIUS_M = 5.0              # walks whose camera paths come this close (meters, 3D) share a location group

# Selection
CANDIDATES = 100000                  # random candidate splits tried (each fills validation with random location groups)
NORMALIZATION_CANDIDATES = 20000     # each distance is divided by its median over this many random splits (rules ignored)
SWAP_STARTS = 5                      # the swaps start from this many of the best random candidates
SEED = 42                            # same seed = same split every time

# Optional: a list of walk ids to CHECK instead of choosing one (None = choose with the rules above)
FIXED_VALIDATION_WALKS = None

for path in (INDEX_FILE, CLASS_PIXELS_FILE):
    print(f"{'found  ' if path.exists() else 'MISSING'} {path}")

## 2. Imports

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 220)

## 3. Frames, walks and location groups

In [ ]:
index = pd.read_csv(INDEX_FILE, dtype={"frame_id": str})
pixels = pd.read_csv(CLASS_PIXELS_FILE)
frames = index.merge(pixels, on="mask", how="left", validate="one_to_one")
all_class_names = [c for c in pixels.columns if c not in ("mask", "ids_above_30", "camera")]
class_names = [c for c in all_class_names if c not in IGNORED_CLASSES]
assert frames[class_names].notna().all().all(), "some frames have no class pixels"

walks = frames.groupby("session").agg(camera=("camera", "first"), frames=("frame_id", "size"))
walks["frame_rate"] = walks["frames"].map(FRAMES_TO_FPS)
assert walks["frame_rate"].notna().all(), "a walk has a frame count not in FRAMES_TO_FPS"
sessions = list(walks.index)
print(f"{len(frames):,} frames in {len(sessions)} walks | {len(class_names)} classes used "
      f"(left out: {', '.join(IGNORED_CLASSES)})")
display(walks.groupby(["camera", "frame_rate"]).agg(walks=("frames", "size"), frames=("frames", "sum")))

In [ ]:
# Location groups: link two walks when any two of their camera positions are within LOCATION_RADIUS_M (3D);
# a group = walks connected through such links (union-find). A group is assigned to validation or train as a whole.
points = frames[["pos_x", "pos_y", "pos_z"]].to_numpy()
walk_of_point = frames["session"].map({session: number for number, session in enumerate(sessions)}).to_numpy()
parent = list(range(len(sessions)))


def find(walk):
    """Root of a walk's group (with path halving)."""
    while parent[walk] != walk:
        parent[walk] = parent[parent[walk]]
        walk = parent[walk]
    return walk


for first, second in cKDTree(points).query_pairs(LOCATION_RADIUS_M):
    a, b = walk_of_point[first], walk_of_point[second]
    if a != b:
        parent[find(a)] = find(b)
roots = [find(walk) for walk in range(len(sessions))]
group_ids = {root: number for number, root in enumerate(sorted(set(roots)))}
walks["group"] = [group_ids[root] for root in roots]
group_sizes = walks["group"].value_counts()
groups_with_both = sum(walks.loc[walks["group"] == g, "camera"].nunique() > 1 for g in group_sizes.index)
print(f"{len(group_sizes)} location groups (radius {LOCATION_RADIUS_M} m) | walks alone: {int((group_sizes == 1).sum())} | "
      f"largest group: {int(group_sizes.max())} walks ({group_sizes.max() / len(sessions):.1%}) | "
      f"groups with both cameras: {groups_with_both}")
figure, axis = plt.subplots(figsize=(10, 3))
axis.hist(group_sizes, bins=range(1, int(group_sizes.max()) + 2), log=True)
axis.set_xlabel("walks per location group"); axis.set_ylabel("groups (log)"); axis.set_title("Location group sizes")
plt.show()

## 4. One row per walk, per camera

In [ ]:
cameras = ["chest", "head"]
camera_of_walk = walks["camera"].to_numpy()
walk_frames = walks["frames"].to_numpy()
walk_rate = walks["frame_rate"].to_numpy()
walk_group = walks["group"].to_numpy()
per_walk = frames.groupby("session")
walk_class_pixels = per_walk[class_names].sum().reindex(sessions).to_numpy(dtype=np.float64)       # walks x classes
walk_class_frames = (frames[class_names] > 0).groupby(frames["session"]).sum().reindex(sessions).to_numpy()
rates = sorted(FRAMES_TO_FPS.values())

required, max_walks_per_class, fps_target = {}, {}, {}
for camera in cameras:
    mine = camera_of_walk == camera
    class_walk_counts = (walk_class_pixels[mine] > 0).sum(axis=0)
    required[camera] = class_walk_counts >= REQUIRED_CLASS_MIN_WALK_SHARE * mine.sum()
    max_walks_per_class[camera] = np.floor(class_walk_counts * MAX_SHARE_OF_CLASS_WALKS)
    fps_target[camera] = {rate: VALIDATION_SHARE * int(((walk_rate == rate) & mine).sum()) for rate in rates}
    print(f"{camera}: {mine.sum()} walks | classes that must appear in validation: {int(required[camera].sum())} of "
          f"{len(class_names)} | target validation walks per frame rate: "
          + ", ".join(f"{rate} FPS {target:.1f}" for rate, target in fps_target[camera].items()))

## 5. Choose the validation walks

Same scoring as the chest split, computed **per camera** and added up: class pixels (half the summed absolute
difference of the pixel shares), class frames (mean absolute difference of the share of frames containing each class),
frame-rate mix, and size (distance from 10%). Each distance is divided by its median over random splits before adding.
A candidate is made by adding random location groups until both cameras reach 10% of their frames.

In [ ]:
def camera_distances(is_validation, camera):
    """Distances between validation and train for one camera. Returns {name: float}."""
    mine = camera_of_walk == camera
    validation, train = is_validation & mine, ~is_validation & mine
    validation_frames, train_frames = walk_frames[validation].sum(), walk_frames[train].sum()
    validation_pixels, train_pixels = walk_class_pixels[validation].sum(axis=0), walk_class_pixels[train].sum(axis=0)
    rate_share = lambda part: np.array([(walk_rate[part] == rate).sum() for rate in rates]) / max(part.sum(), 1)
    return {
        "class pixels": 0.5 * np.abs(validation_pixels / validation_pixels.sum() - train_pixels / train_pixels.sum()).sum(),
        "class frames": np.abs(walk_class_frames[validation].sum(axis=0) / validation_frames
                               - walk_class_frames[train].sum(axis=0) / train_frames).mean(),
        "frame rates": np.abs(rate_share(validation) - rate_share(train)).mean(),
        "size": abs(validation_frames / walk_frames[mine].sum() - VALIDATION_SHARE),
    }


def split_distances(is_validation):
    """Both cameras' distances, named "<camera> <distance>". Returns {name: float}."""
    return {f"{camera} {name}": value for camera in cameras for name, value in camera_distances(is_validation, camera).items()}


def broken_rules(is_validation):
    """The rules a candidate breaks (empty list = all rules kept)."""
    broken = []
    for camera in cameras:
        mine = camera_of_walk == camera
        validation, train = is_validation & mine, ~is_validation & mine
        share = walk_frames[validation].sum() / walk_frames[mine].sum()
        if not FRAME_SHARE_RANGE[0] <= share <= FRAME_SHARE_RANGE[1]:
            broken.append(f"{camera}: validation holds {share:.1%} of the frames")
        for rate, target in fps_target[camera].items():
            count = int((walk_rate[validation] == rate).sum())
            if abs(count - target) > max(1.0, FPS_WALK_TOLERANCE * target):
                broken.append(f"{camera}: {count} validation walks at {rate} FPS (target {target:.1f})")
        validation_class_frames = walk_class_frames[validation].sum(axis=0)
        train_class_frames = walk_class_frames[train].sum(axis=0)
        validation_count, train_count = walk_frames[validation].sum(), walk_frames[train].sum()
        validation_class_walks = (walk_class_pixels[validation] > 0).sum(axis=0)
        for index, class_name in enumerate(class_names):
            validation_share = validation_class_frames[index] / max(validation_count, 1)
            train_share = train_class_frames[index] / max(train_count, 1)
            if required[camera][index]:
                if validation_class_frames[index] < MIN_FRAMES_IN_VALIDATION:
                    broken.append(f"{camera} {class_name}: only {validation_class_frames[index]} validation frames")
                if not MIN_RELATIVE_FRAME_SHARE * train_share <= validation_share <= MAX_RELATIVE_FRAME_SHARE * train_share:
                    broken.append(f"{camera} {class_name}: in {validation_share:.1%} of validation frames, "
                                  f"{train_share:.1%} in train")
            if validation_class_walks[index] > max_walks_per_class[camera][index]:
                broken.append(f"{camera} {class_name}: {validation_class_walks[index]} validation walks "
                              f"(max {int(max_walks_per_class[camera][index])})")
    return broken


group_members = [np.flatnonzero(walk_group == group) for group in range(walk_group.max() + 1)]
camera_totals = {camera: walk_frames[camera_of_walk == camera].sum() for camera in cameras}


def random_candidate(generator):
    """Add location groups in random order until both cameras hold VALIDATION_SHARE of their frames; skip a group that
    would push a camera above FRAME_SHARE_RANGE. Returns yes/no per walk."""
    is_validation = np.zeros(len(sessions), dtype=bool)
    filled = {camera: 0 for camera in cameras}
    for group in generator.permutation(len(group_members)):
        members = group_members[group]
        added = {camera: walk_frames[members][camera_of_walk[members] == camera].sum() for camera in cameras}
        if any(added[c] and filled[c] / camera_totals[c] >= VALIDATION_SHARE for c in cameras):
            continue                                      # a camera that is already full would grow
        if any((filled[c] + added[c]) / camera_totals[c] > FRAME_SHARE_RANGE[1] for c in cameras):
            continue
        is_validation[members] = True
        for c in cameras:
            filled[c] += added[c]
        if all(filled[c] / camera_totals[c] >= VALIDATION_SHARE for c in cameras):
            break
    return is_validation

In [ ]:
if FIXED_VALIDATION_WALKS is not None:
    is_chosen = np.isin(sessions, FIXED_VALIDATION_WALKS)
    print("Checking the fixed list:", broken_rules(is_chosen) or "all rules kept")
    candidates_kept = None
else:
    generator = np.random.default_rng(SEED)
    kept, kept_distances, reference = [], [], []
    start_time = time.time()
    for candidate in range(1, CANDIDATES + 1):
        # random_candidate takes: the random generator; returns: yes/no per walk (validation)
        is_validation = random_candidate(generator)
        if candidate <= NORMALIZATION_CANDIDATES:
            reference.append(split_distances(is_validation))
        if not broken_rules(is_validation):
            kept.append(is_validation)
            kept_distances.append(split_distances(is_validation))
        if candidate % 10000 == 0 or candidate == CANDIDATES:
            elapsed = time.time() - start_time
            print(f"[{candidate:,}/{CANDIDATES:,}] kept {len(kept):,} | elapsed {elapsed:.0f}s | "
                  f"ETA {elapsed / candidate * (CANDIDATES - candidate):.0f}s", flush=True)
    candidates_kept = len(kept)
    if not kept:
        raise RuntimeError("no candidate kept all rules: look at broken_rules(random_candidate(...)) and the settings")
    medians = pd.DataFrame(reference).median().replace(0, 1e-12)
    distances = pd.DataFrame(kept_distances)
    distances["total"] = (distances / medians).sum(axis=1)
    print(f"\n{candidates_kept:,} of {CANDIDATES:,} candidates kept all rules; best total "
          f"{distances['total'].min():.2f} (median {distances['total'].median():.2f})")
    display(distances.sort_values("total").head(5).round(4))

    def total_distance(is_validation):
        """Each distance divided by its median over random splits, added up."""
        return sum(value / medians[name] for name, value in split_distances(is_validation).items())

    def improve_by_swaps(start):
        """Each round: try every (validation group, train group) swap; keep the best one that lowers the total and
        keeps all rules; stop when no swap helps. Returns (split, total, swaps made)."""
        current, current_total, swaps = start.copy(), total_distance(start), 0
        while True:
            validation_groups = sorted(set(walk_group[current]))
            train_groups = sorted(set(walk_group[~current]))
            best_swap, best_total = None, current_total
            for out_group in validation_groups:
                for in_group in train_groups:
                    trial = current.copy()
                    trial[group_members[out_group]] = False
                    trial[group_members[in_group]] = True
                    # cheap size check first (most swaps break it), then the full rules
                    if any(not FRAME_SHARE_RANGE[0] <= walk_frames[trial & (camera_of_walk == c)].sum() / camera_totals[c]
                           <= FRAME_SHARE_RANGE[1] for c in cameras):
                        continue
                    if broken_rules(trial):
                        continue
                    trial_total = total_distance(trial)
                    if trial_total < best_total - 1e-12:
                        best_swap, best_total = (out_group, in_group), trial_total
            if best_swap is None:
                return current, current_total, swaps
            current[group_members[best_swap[0]]] = False
            current[group_members[best_swap[1]]] = True
            current_total, swaps = best_total, swaps + 1

    best_result = None
    start_time = time.time()
    order = distances["total"].sort_values().index[:SWAP_STARTS]
    print(f"\nSwaps from the {len(order)} best random candidates:")
    for number, candidate_index in enumerate(order, start=1):
        # improve_by_swaps takes: a starting split; returns: (improved split, its total, swaps made)
        improved, improved_total, swaps = improve_by_swaps(kept[candidate_index])
        if best_result is None or improved_total < best_result[1] - 1e-12:
            best_result = (improved, improved_total)
        elapsed = time.time() - start_time
        print(f"  [{number}/{len(order)}] {distances.loc[candidate_index, 'total']:.3f} -> {improved_total:.3f} after "
              f"{swaps} swaps | best {best_result[1]:.3f} | elapsed {elapsed:.0f}s | "
              f"ETA {elapsed / number * (len(order) - number):.0f}s", flush=True)
    is_chosen = best_result[0]
chosen_walks = sorted(np.array(sessions)[is_chosen].tolist())
assert not broken_rules(is_chosen), broken_rules(is_chosen)
print(f"\nchosen: {len(chosen_walks)} validation walks in {len(set(walk_group[is_chosen]))} location groups")

### Save the chosen walks

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "validation_walks.txt").write_text("\n".join(chosen_walks) + "\n")
record = {
    "validation_walks": chosen_walks,
    "per_camera": {camera: {"validation_walks": int((is_chosen & (camera_of_walk == camera)).sum()),
                            "validation_frames": int(walk_frames[is_chosen & (camera_of_walk == camera)].sum()),
                            "train_frames": int(walk_frames[~is_chosen & (camera_of_walk == camera)].sum())}
                   for camera in cameras},
    "location_groups_in_validation": int(len(set(walk_group[is_chosen]))),
    "distances": {name: round(float(value), 5) for name, value in split_distances(is_chosen).items()},
    "candidates_tried": None if FIXED_VALIDATION_WALKS is not None else CANDIDATES,
    "candidates_kept": candidates_kept,
    "settings": {"VALIDATION_SHARE": VALIDATION_SHARE, "FRAME_SHARE_RANGE": FRAME_SHARE_RANGE,
                 "FRAMES_TO_FPS": {str(k): v for k, v in FRAMES_TO_FPS.items()}, "FPS_WALK_TOLERANCE": FPS_WALK_TOLERANCE,
                 "REQUIRED_CLASS_MIN_WALK_SHARE": REQUIRED_CLASS_MIN_WALK_SHARE,
                 "MIN_FRAMES_IN_VALIDATION": MIN_FRAMES_IN_VALIDATION, "MIN_RELATIVE_FRAME_SHARE": MIN_RELATIVE_FRAME_SHARE,
                 "MAX_RELATIVE_FRAME_SHARE": MAX_RELATIVE_FRAME_SHARE, "MAX_SHARE_OF_CLASS_WALKS": MAX_SHARE_OF_CLASS_WALKS,
                 "IGNORED_CLASSES": IGNORED_CLASSES, "LOCATION_RADIUS_M": LOCATION_RADIUS_M, "CANDIDATES": CANDIDATES,
                 "NORMALIZATION_CANDIDATES": NORMALIZATION_CANDIDATES, "SWAP_STARTS": SWAP_STARTS, "SEED": SEED,
                 "FIXED_VALIDATION_WALKS": FIXED_VALIDATION_WALKS},
}
(RESULTS_DIR / "validation_split.json").write_text(json.dumps(record, indent=2))
print("saved", RESULTS_DIR / "validation_walks.txt", "and", RESULTS_DIR / "validation_split.json")

## 6. Comparison: size and frame rates

In [ ]:
rows = []
for camera in cameras:
    for rate in rates + ["all"]:
        mine = (camera_of_walk == camera) & ((walk_rate == rate) if rate != "all" else True)
        rows.append({"camera": camera, "frame rate": rate,
                     "validation walks": int((is_chosen & mine).sum()), "train walks": int((~is_chosen & mine).sum()),
                     "validation frames": int(walk_frames[is_chosen & mine].sum()),
                     "train frames": int(walk_frames[~is_chosen & mine].sum())})
size_table = pd.DataFrame(rows).set_index(["camera", "frame rate"])
size_table["validation share of frames %"] = (100 * size_table["validation frames"]
                                              / (size_table["validation frames"] + size_table["train frames"])).round(1)
display(size_table)

## 7. Comparison: classes (per camera)

In [ ]:
for camera in cameras:
    mine = camera_of_walk == camera
    validation, train = is_chosen & mine, ~is_chosen & mine
    table = pd.DataFrame({
        "train pixel %": 100 * walk_class_pixels[train].sum(axis=0) / walk_class_pixels[train].sum(),
        "validation pixel %": 100 * walk_class_pixels[validation].sum(axis=0) / walk_class_pixels[validation].sum(),
        "train frames %": 100 * walk_class_frames[train].sum(axis=0) / walk_frames[train].sum(),
        "validation frames %": 100 * walk_class_frames[validation].sum(axis=0) / walk_frames[validation].sum(),
        "must appear": required[camera]}, index=class_names).round(3)
    table = table[(table["train pixel %"] > 0) | (table["validation pixel %"] > 0)]
    print(f"\n{camera}: classes present in its synthetic walks")
    display(table.sort_values("train pixel %", ascending=False))

## 8. Check: no validation walk retraces a train walk's spot

In [ ]:
validation_points = points[is_chosen[walk_of_point]]
train_points = points[~is_chosen[walk_of_point]]
# cKDTree.query takes: points; returns: (distance to the nearest train position, its index) for each validation position
nearest, _ = cKDTree(train_points).query(validation_points)
print(f"closest distance between any validation position and any train position: {nearest.min():.2f} m "
      f"(must be > {LOCATION_RADIUS_M} m) | median {np.median(nearest):.1f} m")
assert nearest.min() > LOCATION_RADIUS_M

## 9. Summary

Fill in after running: validation walks and frames per camera, frame-rate mix, classes, the location check. Then run
`../prepare_sanpo_synthetic_modal.py` again on each session: it skips every finished walk and only rewrites the lists,
now with `val.txt` and `data.yaml`.